In [ ]:
pip install faiss-cpu nltk lxml wordninja

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 541.6/541.6 kB 15.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for wordninja: filename=wordninja-2.0.0-py3-none-any.whl size=541530 sha256=0ad3bcb0224d0959a3c9cd0433bce4996a75e8bdd0547f3df4cb46f7ed3a9199
  Stored in directory: /root/.cache/pip/wheels/6e/31/92/f12667e4dd102e546832a02f41feca39ae916889006517e595
Successfully built wordninja


In [ ]:
from sentence_transformers import SentenceTransformer
import wordninja
import faiss
import lxml
import json
import re
import numpy as np
import pickle
import os
import time
from tensorflow.keras.utils import register_keras_serializable
from tensorflow.keras.optimizers import AdamW
import pandas as pd
import requests
from bs4 import BeautifulSoup
import nltk
import tensorflow as tf
from tensorflow.keras.models import load_model
nltk.download('punkt_tab')
nltk.download("stopwords")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
url = f"https://www.nationalgeographic.com/animals/mammals/facts/giraffe"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36"
}
response = requests.get(url, headers = headers)
print(response.status_code)

200


In [ ]:
soup = BeautifulSoup(response.text, 'lxml')

In [ ]:
paragraph = ""
paragraphs = soup.find_all('p')
for p in paragraphs:
    paragraph+=p.text

In [ ]:
paragraph = nltk.tokenize.sent_tokenize(paragraph)

In [ ]:
cleaned_sentences = [re.sub(r"\[\d+\]", "", s).strip().replace("\xa0", " ").strip() for s in paragraph]

In [ ]:
cleaned_sentences

["Giraffes are the world's tallest mammals, thanks to their towering legs and long necks.",
 "A giraffe's legs alone are taller than many humans—about 6 feet .",
 'These long legs allow giraffes to run as fast as 35 miles an hour over short distances and cruise comfortably at 10 miles an hour over longer distances.Typically, these fascinating animals roam the open grasslands in small groups of about half a dozen.Bulls sometimes battle one another by butting their long necks and heads.',
 "Such contests aren't usually dangerous and end when one animal submits and walks away.Giraffes use their height to good advantage and browse on leaves and buds in treetops that few other animals can reach (acacias are a favorite).",
 "Even the giraffe's tongue is long!",
 'The 21-inch tongue helps them pluck tasty morsels from branches.',
 'Giraffes eat most of the time and, like cows, regurgitate food and chew it as cud.',
 "A giraffe eats hundreds of pounds of leaves each week and must travel miles 

In [ ]:
@register_keras_serializable(package="Custom", name="WarmUpLearningRate")
class WarmUpLearningRate(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, initial_lr, warmup_steps, decay_schedule_fn):
        super().__init__()
        self.initial_lr = initial_lr
        self.warmup_steps = warmup_steps
        self.decay_schedule_fn = decay_schedule_fn
    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warmup_lr = self.initial_lr * (step / tf.cast(self.warmup_steps, tf.float32))
        return tf.cond(
            step < self.warmup_steps,
            lambda: warmup_lr,
            lambda: self.decay_schedule_fn(step - self.warmup_steps)
        )
    def get_config(self):
        return {
            "initial_lr": self.initial_lr,
            "warmup_steps": self.warmup_steps,
            "decay_schedule_fn": tf.keras.optimizers.schedules.serialize(self.decay_schedule_fn),
        }
    @classmethod
    def from_config(cls, config):
        config["decay_schedule_fn"] = tf.keras.optimizers.schedules.deserialize(config["decay_schedule_fn"])
        return cls(**config)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor = 'val_loss',
    patience = 10,
    restore_best_weights= True
)
decay_schedule = tf.keras.optimizers.schedules.PolynomialDecay(
    initial_learning_rate=0.001,
    decay_steps=10000,
    end_learning_rate=1e-5,
    power=1.05
)
warmup_lr = WarmUpLearningRate(
    initial_lr=0.001,
    warmup_steps=1000,
    decay_schedule_fn=decay_schedule
)
optimizer = AdamW(
    learning_rate=warmup_lr,
    weight_decay=1e-4
)
embedding_model = SentenceTransformer("multi-qa-MiniLM-L6-cos-v1")  # or "all-mpnet-base-v2"
EMBED_DIM = embedding_model.get_sentence_embedding_dimension()
print("embed dim:", EMBED_DIM)
ANN = load_model("/content/NER_Model_1.keras")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

embed dim: 384


In [ ]:
def sentence_to_embedding(sentence, model, embedding_dim=384):
    words = nltk.word_tokenize(sentence)
    stop = nltk.corpus.stopwords.words('english')
    words = [word for word in words if word.lower() not in stop and word.isalnum()]
    if not words:
        return np.zeros((1, embedding_dim))
    embeddings = model.encode(words)
    if embeddings.ndim == 1:
        embeddings = embeddings[np.newaxis, :]
    mean_embedding = embeddings.mean(axis=0)
    return mean_embedding.reshape(1, -1)

LABELS = ['Adaptation', 'Behavior', 'Diet', 'Feature', 'Genus',
          'Gpe', 'Habitat', 'Roleinecosystem', 'Threat', 'Timeperiod']

def predict_label(sentence, model, ANN):
    embedding = sentence_to_embedding(sentence, model)
    pred = ANN.predict(embedding)
    pred_label_idx = np.argmax(pred, axis=1)
    return LABELS[pred_label_idx[0]]

end_sp = pd.read_csv("/content/endangered.csv")
end_sp = end_sp.drop(columns='Unnamed: 1', axis=1)
end_sp = np.array(end_sp).reshape(1, -1)

def detect_status(sentence):
    sentence = sentence.lower()
    for word in end_sp[0]:
        if word in sentence:
            return True
    return False

def get_SN():
  names = []
  count = 0
  for tag in soup.find_all(["i", "em", "b"]):
    if count == 6:
      break
    txt = tag.get_text(" ", strip=True)
    if txt and " " in txt:
        names.append(txt)
        count += 1
  return names

In [ ]:
names = []
count = 0
for tag in soup.find_all(["i", "em", "b"]):
  if count == 6:
    break
  txt = tag.get_text(" ", strip=True)
  if txt and " " in txt:
      names.append(txt)
      count += 1
print(names)

['Giraffa camelopardalis reticulata', 'Giraffa camelopardalis rothschildi']


In [ ]:
animal_name = input()
wikipedia_url = f"https://en.wikipedia.org/wiki/{animal_name}"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36"
}
response = requests.get(url, headers = headers)
if(response.status_code == 403):
  print("ERROR")
soup = BeautifulSoup(response.text, 'html.parser')
paragraph = ""
paragraphs = soup.find_all('p')
for p in paragraphs:
    paragraph+=p.text
paragraph = nltk.tokenize.sent_tokenize(paragraph)
cleaned_sentences = [re.sub(r"\[\d+\]", "", s).strip().replace("\xa0", " ").strip() for s in paragraph[0:130]]
names = get_SN()
count = 1
animal_card = {
        "id" : f"{animal_name}_{count}",
        "Name" : animal_name,
        "scientific_name" : names,
        "facts" : [],
        "source" : "Wikipedia"
    }
status_sentences = [s for s in cleaned_sentences if detect_status(s)]
for i in status_sentences:
    D = {"label" : "Status" , "text" : ""}
    D["text"] = i
    animal_card["facts"].append(D)
for i, sentence in enumerate(cleaned_sentences):  # first 20 sentences only
        pred = predict_label(sentence, embedding_model, ANN)
        D = {"label" : pred , "text" : ""}
        if D["text"] == "":
            D["text"] = sentence
        else:
            D["text"] += " " + sentence  # concatenate multiple
        animal_card["facts"].append(D)

Tiger
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step


In [ ]:
cleaned_sentences

["Giraffes are the world's tallest mammals, thanks to their towering legs and long necks.",
 "A giraffe's legs alone are taller than many humans—about 6 feet .",
 'These long legs allow giraffes to run as fast as 35 miles an hour over short distances and cruise comfortably at 10 miles an hour over longer distances.Typically, these fascinating animals roam the open grasslands in small groups of about half a dozen.Bulls sometimes battle one another by butting their long necks and heads.',
 "Such contests aren't usually dangerous and end when one animal submits and walks away.Giraffes use their height to good advantage and browse on leaves and buds in treetops that few other animals can reach (acacias are a favorite).",
 "Even the giraffe's tongue is long!",
 'The 21-inch tongue helps them pluck tasty morsels from branches.',
 'Giraffes eat most of the time and, like cows, regurgitate food and chew it as cud.',
 "A giraffe eats hundreds of pounds of leaves each week and must travel miles 

In [ ]:
animal_card

{'id': 'Tiger_1',
 'Name': 'Tiger',
 'scientific_name': ['Giraffa camelopardalis reticulata',
  'Giraffa camelopardalis rothschildi'],
 'facts': [{'label': 'Status',
   'text': "To do so they must spread their legs and bend down in an awkward position that makes them vulnerable to predators like Africa's big cats."},
  {'label': 'Status',
   'text': 'In 2016, some scientists released a study that claims genetic differences among giraffe populations indicate the existence of four distinct giraffe species.WATCH: These Rare Giraffes Were Killed Just for Their TailsCopyright © 1996-2015 National Geographic SocietyCopyright © 2015-2025 National Geographic Partners, LLC.'},
  {'label': 'Feature',
   'text': "Giraffes are the world's tallest mammals, thanks to their towering legs and long necks."},
  {'label': 'Feature',
   'text': "A giraffe's legs alone are taller than many humans—about 6 feet ."},
  {'label': 'Feature',
   'text': 'These long legs allow giraffes to run as fast as 35 miles 

In [ ]:

# ========== 1️⃣ Wikipedia Scraper ==========
def web1(animal_name):
    # Convert e.g. "tiger quoll" → "Tiger_quoll"
    animal_name = "_".join(wordninja.split(animal_name.strip()))
    animal_name = animal_name.capitalize()

    url = f"https://en.wikipedia.org/wiki/{animal_name}"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Chrome/120.0.0.0 Safari/537.36"
    }

    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        print(f"⚠️ Error fetching {url}")
        return [], []

    soup = BeautifulSoup(response.text, 'html.parser')
    paragraphs = soup.find_all('p')
    text = " ".join(p.text for p in paragraphs)

    sentences = nltk.tokenize.sent_tokenize(text)
    cleaned_sentences = [
        re.sub(r"\[\d+\]", "", s).strip().replace("\xa0", " ").strip()
        for s in sentences[:130]
    ]

    # Try to extract scientific names from italic tags
    names = []
    count = 0
    for tag in soup.find_all(["i", "em", "b"]):
        if count == 6:
            break
        txt = tag.get_text(" ", strip=True)
        if txt and " " in txt:
            names.append(txt)
            count += 1

    return cleaned_sentences, names


# ========== 2️⃣ A-Z Animals Scraper ==========
def web2(animal_name):
    # Convert e.g. "Tiger quoll" → "tiger+quoll"
    animal_query = "+".join(wordninja.split(animal_name.strip().lower()))

    url = f"https://a-z-animals.com/?s={animal_query}"
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        )
    }

    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        print(f"⚠️ Error fetching search page for {animal_name}")
        return []

    soup = BeautifulSoup(response.text, "html.parser")
    articles = soup.find_all("article")
    print("Found articles:", len(articles))
    rec = {}
    for article in articles:
        aria_label = article.get("aria-label")
        links = article.find_all("a", href=True)
        if aria_label and links:
            rec[aria_label] = links[0]["href"]

    combined_sent = []
    count = 0
    for title, link in rec.items():
        if count == 5:
            break
        sub_resp = requests.get(link, headers=headers)
        if sub_resp.status_code != 200:
            continue
        sub_soup = BeautifulSoup(sub_resp.text, "html.parser")
        paragraphs = sub_soup.find_all("p")
        text = ""
        for p in paragraphs:
            if (
                re.search(r"[\(\)\[\]\{\}\<\>]", p.text)
                or p.text.strip().startswith("©")
                or "Shutterstock" in p.text
            ):
                continue
            text += p.text.strip() + " "
        tok_sent = nltk.tokenize.sent_tokenize(text)
        if len(tok_sent) > 6:
            selected_sent = tok_sent[2:-4]
        else:
            selected_sent = tok_sent
        combined_sent.extend(selected_sent)
        count += 1
    print("Combined sentences:", len(combined_sent))
    return combined_sent

# ========== 3️⃣ Animal Card Builder ==========
def build_animal_card(animal_name, scientific_name, sentences, source):
    animal_card = {
        "id": f"{animal_name}_{source}",
        "Name": animal_name,
        "scientific_name": scientific_name,
        "facts": [],
        "source": source,
    }

    # Extract status-related sentences
    if(source == "Wikipedia"):
      status_sentences = [s for s in sentences if detect_status(s)]
      for s in status_sentences:
          animal_card["facts"].append({"label": "Status", "text": s})

    # Predict label for remaining sentences
    for sentence in sentences:
        pred = predict_label(sentence, embedding_model, ANN)
        animal_card["facts"].append({"label": pred, "text": sentence})

    return animal_card


# ========== 4️⃣ MAIN EXECUTION ==========
def main():
    animal_name = input("Enter animal name: ").strip()

    data_1, scientific_name = web1(animal_name)
    data_2 = web2(animal_name)

    animal_card_1 = build_animal_card(animal_name, scientific_name, data_1, "Wikipedia")
    animal_card_2 = build_animal_card(animal_name, scientific_name, data_2, "A-Z Animals")

    return animal_card_1, animal_card_2


# Run
animal_card_1, animal_card_2 = main()

print("\n🐾 ANIMAL CARD 1 (Wikipedia):")
print(animal_card_1)
print("\n🐾 ANIMAL CARD 2 (A-Z Animals):")
print(animal_card_2)

Enter animal name: Tiger
Found articles: 50
Combined sentences: 0
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━

KeyboardInterrupt: 

In [ ]:
animal_card_1

{'id': 'tiger_quoll_Wikipedia',
 'Name': 'tiger_quoll',
 'scientific_name': ['Viverra maculata'],
 'facts': [{'label': 'Status',
   'text': 'The species is listed as near-threatened on the IUCN Red List, and is primarily threatened by habitat loss caused by human activities.'},
  {'label': 'Status',
   'text': 'Tiger quolls are rare in southeastern Queensland and mainly restricted to national parks.'},
  {'label': 'Status',
   'text': 'In Victoria, quoll populations have declined by nearly 50%.'},
  {'label': 'Status',
   'text': 'The range decline was not as severe in New South Wales, but they are still rare.'},
  {'label': 'Status',
   'text': "The quoll was probably never very numerous in South Australia, but although considered locally extinct for 130 years, one was captured in the state's south-east in 2023."},
  {'label': 'Status',
   'text': 'Tiger quolls were once native to Flinders Island and King Island, but are locally extinct (extirpated) since the 20th century, so are not 

In [ ]:
animal = input("Enter animal name: ").strip()
animal = "_".join(wordninja.split(animal))
animal.lower().capitalize()

Enter animal name: tiger quoll


'Tiger_quoll'